In [1]:
import os
import h5py
import numpy as np
from PIL import Image
from collections import defaultdict


def GetRGBDepthPathPairs(base):
    u = defaultdict(dict)
    # For each file in the directory
    for root, dirs, files in os.walk(base):
        for file in files:
            filepath = os.path.join(root, file)
            res = filepath.split('/')
            # Create a dict key to uniquely identify frames
            r1 = res[3] + '_' + res[4] + '_' + res[-1].split('.')[1]

            # Assumption is for every rgb there is a depth
            if file.lower().endswith(('.hdf5', '.hdf')):
                u[r1]['depth'] = filepath
            elif file.lower().endswith(('.jpg', '.jpeg')):
                u[r1]['rgb'] = filepath

    # Convert the default dict to a list
    pairs_list = []
    # Here implicitly it will be checked if every pair has a valid depth and rgb path, else there should be an error
    for k, v in u.items():
        pairs_list.append({
            'rgb': v['rgb'],
            'depth': v['depth']
        })

    return pairs_list


base = './Hypersim/'
train = base + 'Train/'
test = base + 'Test/'

train_pairs = GetRGBDepthPathPairs(train)
test_pairs = GetRGBDepthPathPairs(test)

In [2]:
print(len(train_pairs), len(test_pairs))

2000 100


In [6]:
depth_file = train_pairs[0]['depth']
rgb_file = train_pairs[0]['rgb']

# Load depth
with h5py.File(depth_file, 'r') as f:
    print("Keys:", list(f.keys()))

    # Replace with the correct dataset key if needed
    depth = f[list(f.keys())[0]][:]

# Load RGB
rgb = np.array(Image.open(rgb_file))

Keys: ['dataset']


In [7]:
img_uint8 = (rgb).clip(0, 255).astype(np.uint8)

img = Image.fromarray(img_uint8)
img.save('./resources/hypersim_rgb.png')

In [8]:
img = depth
img = img * 10000.0 
img = img.astype(np.uint16)
img = Image.fromarray(img)
img.save('./resources/hypersim_depth.png')

/tmp/ipykernel_64483/1985932730.py:2: RuntimeWarning: overflow encountered in multiply
  img = img * 10000.0
/tmp/ipykernel_64483/1985932730.py:3: RuntimeWarning: invalid value encountered in cast
  img = img.astype(np.uint16)
